# Automatización WB — Fase 4 (Consolidación): Matriz real + CUADRO FINAL

Equivalente al notebook NB (`Automatizacion_Fase4_Consolidacion.ipynb`), para las flotas WB
**B767** y **B787** — dos procesos separados (`FLOTA_WB = "767"` o `"787"`).

Continúa donde quedó `Automatizacion_WB_Fase3_Asignacion_Instructor_Vuelos.ipynb` (mismo query
+ mismo emparejamiento Matriz↔pairings reales). Este notebook agrega, cada uno detrás de su
propia vista previa:

1. **Reemplazar el placeholder en la Matriz** (`"LCK B767"`/`"LCK B787"`) por el texto real del
   vuelo — y, cuando el pairing es LIM-MIA-LIM (ida y vuelta en fechas distintas), escribir
   **también** la celda de la fecha de vuelta (confirmado por Fernando el 2026-09-24: ambos
   días deben quedar marcados).
2. **CUADRO FINAL** en Archivo 10 (hoja "prueba de LCK 767" / "prueba de LCK 787") — Fecha,
   Día, Vuelo, Ruta, Cupos, INS (Grupo se deja en blanco, igual regla que NB).

**Todavía NO incluye** (falta información para no inventar nada — ver la última sección):
- La segunda tabla del Freeze (falta el link/gid de las pestañas "LCK 767"/"LCK B787" del
  archivo "202609 Freeze LP").
- El cruce "INS F a considerar"/"Grupo" por tripulante (Y→Z/AA): en NB los grupos se leen de 4
  celdas fijas (`AD2..AG2`); para WB el layout que se ve en tu captura (`AQ`/`AR`/`AS`/`AT`/`AU`/`AV`
  con pares "Grupo 1"/"Karla y Cris"/"Grupo 2"/"Sebas"/...) parece distinto y no tengo las
  celdas exactas confirmadas todavía.

**Importante sobre las columnas del CUADRO FINAL:** confirmaste que el orden de columnas es
DISTINTO entre las dos flotas (767: fecha, día, ruta, vuelo, cupo, INS, Grupo · 787: Fecha,
DíaSEM, Vuelo, Ruta, N° Cupos, Grupos, INS) — por eso este notebook busca cada columna por su
nombre de encabezado (igual patrón que Archivo 9), no por una posición fija, así funciona para
ambas sin necesitar código distinto por flota.


## 1. Instalar dependencias y autenticar

In [ ]:
!pip install -q gspread


In [ ]:
import unicodedata
import collections
from collections import defaultdict
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
client = bigquery.Client(project="datadem-home")
print("Autenticado: BigQuery + Google Sheets con tu cuenta.")


## 2. Elegir la flota WB

In [ ]:
FLOTA_WB = "767"  # o "787"

WB_CONFIG = {
    "767": {
        "subfleets": {"763"},
        "actividad_matriz": "LCK B767",
        "cupos_por_vuelo": 5,
        "archivo10_gid": 1025385274,
        "archivo10_tab_esperada": "prueba de LCK 767",
    },
    "787": {
        "subfleets": {"788", "789"},
        "actividad_matriz": "LCK B787",
        "cupos_por_vuelo": 6,
        "archivo10_gid": 16355364,
        "archivo10_tab_esperada": "prueba de LCK 787",
    },
}
if FLOTA_WB not in WB_CONFIG:
    raise ValueError("FLOTA_WB debe ser '767' o '787'")
cfg = WB_CONFIG[FLOTA_WB]
print(f"Flota seleccionada: WB {FLOTA_WB} -> '{cfg['actividad_matriz']}', {cfg['cupos_por_vuelo']} cupos/vuelo")


## 3. Query a BigQuery + candidatos (idéntico a la Fase 3 de WB)

Misma función `cargar_y_filtrar_wb` ya usada y probada en
`Automatizacion_WB_Fase3_Asignacion_Instructor_Vuelos.ipynb`.

In [ ]:
MES_OBJETIVO = 10
ANIO_OBJETIVO = 2026

QUERY_WB = """
SELECT
  pairing_id                       AS trip,
  pairing_start_date               AS fecha_inicio_trip,
  flight_start_date_local_time     AS inicio_vuelo_lt,
  flight_month_description         AS mes,
  duty_calendar_day_number         AS dia_duty,
  flight_number                    AS vuelo,
  departure_airport_code           AS dep,
  arrival_airport_code             AS arr,
  flight_departure_time_crew_base  AS std_hb,
  flight_arrival_hour_block_time   AS sta_hb,
  flight_block_time                AS hbt,
  subfleet_code                    AS sub_fleet,
  is_crew_passenger                AS pax,
  duty_presentation_date_at        AS presentacion_duty_date_lt

FROM `operations-data-prod.carmen_gold.crew_pairing_carmen_system`

WHERE
  flight_start_date_local_time BETWEEN DATE '2026-09-01' AND DATE '2026-10-31'
  AND subsidiary_code IN ('LP')
  AND load_type_code = 'FP'
  AND crew_range_type_code = 'SAB'
  AND subfleet_code IN ('763', '788', '789')

QUALIFY
  CASE
    WHEN load_type_code = 'FP' AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'FP' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    WHEN load_type_code = 'ES' AND
         MAX(CASE WHEN load_type_code = 'FP' THEN 0 ELSE 0 END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year) = -1 AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'ES' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    ELSE -1
  END = 0

ORDER BY pairing_id ASC
"""

df_raw = client.query(QUERY_WB).to_dataframe(create_bqstorage_client=False)
print(f"Filas descargadas (B767+B787): {len(df_raw)}")

ALLOWED_ROUTES = {("LIM", "MIA"), ("MIA", "LIM"), ("LIM", "SCL"), ("SCL", "LIM")}
ALLOWED_FLIGHTS = {2480, 2481, 2695, 2694, 2698, 2699, 2413, 2412, 2697, 2696}
DIAS_OK = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday"}
WD_ES = {"Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
         "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado", "Sunday": "domingo"}
HEADERS_WB = ["Pairing ID", "FECHA REAL", "MES", "Day of Week", "Flight No",
              "Dep Stn", "Arr Stn", "STD", "STA", "DAY", "AC Type", "HBT", "DIA_DUTY"]


def cargar_y_filtrar_wb(df_raw: pd.DataFrame, mes_objetivo: int, anio_objetivo: int) -> pd.DataFrame:
    df = df_raw.copy()
    df["vuelo"] = df["vuelo"].astype(int)
    df["dia_duty"] = df["dia_duty"].astype(int)

    df["inicio_vuelo_lt_dt"] = pd.to_datetime(df["inicio_vuelo_lt"])
    df["presentacion_duty_date_lt_dt"] = pd.to_datetime(df["presentacion_duty_date_lt"])
    df["wd_vuelo"] = df["inicio_vuelo_lt_dt"].dt.day_name()
    df["wd_pres"] = df["presentacion_duty_date_lt_dt"].dt.day_name()
    df["dia_semana"] = df["wd_vuelo"].map(WD_ES)

    df["_pk"] = df["trip"].astype(str) + "|" + df["fecha_inicio_trip"].astype(str)

    df["leg_dia_ok"] = df["wd_vuelo"].isin(DIAS_OK)
    df["leg_pres_ok"] = df["wd_pres"] != "Sunday"
    df["leg_ruta_ok"] = list(zip(df["dep"], df["arr"]))
    df["leg_ruta_ok"] = df["leg_ruta_ok"].isin(ALLOWED_ROUTES) & df["vuelo"].isin(ALLOWED_FLIGHTS)

    g = df.groupby("_pk")
    trip_dia_ok = g["leg_dia_ok"].transform("all")
    trip_pres_ok = g["leg_pres_ok"].transform("all")

    idx_primera_pierna = g["dia_duty"].idxmin()
    primeras_piernas = df.loc[idx_primera_pierna, ["_pk", "inicio_vuelo_lt_dt"]]
    primeras_piernas["mes_ok"] = (primeras_piernas["inicio_vuelo_lt_dt"].dt.month == mes_objetivo) & \
                                  (primeras_piernas["inicio_vuelo_lt_dt"].dt.year == anio_objetivo)
    mapa_mes_ok = primeras_piernas.set_index("_pk")["mes_ok"]
    trip_mes_ok = df["_pk"].map(mapa_mes_ok)
    trip_ruta_ok = g["leg_ruta_ok"].transform("all")

    def dia_duty_contiguo_y_corto(s):
        vals = sorted(s.dropna().unique())
        if not vals:
            return False
        rango_ok = (vals[-1] - vals[0]) <= 2
        contiguo = all(b - a == 1 for a, b in zip(vals, vals[1:]))
        return rango_ok and contiguo

    trip_dia_duty_ok = g["dia_duty"].transform(dia_duty_contiguo_y_corto)

    df["trip_valido"] = trip_dia_ok & trip_mes_ok & trip_pres_ok & trip_ruta_ok & trip_dia_duty_ok

    validos = df[df["trip_valido"]].copy()
    validos["fecha_real_fmt"] = validos["inicio_vuelo_lt_dt"].dt.strftime("%d/%m/%Y")

    validos = validos.rename(columns={
        "trip": "Pairing ID", "fecha_real_fmt": "FECHA REAL", "mes": "MES",
        "dia_semana": "Day of Week", "vuelo": "Flight No", "dep": "Dep Stn", "arr": "Arr Stn",
        "std_hb": "STD", "sta_hb": "STA", "pax": "DAY", "sub_fleet": "AC Type",
        "hbt": "HBT", "dia_duty": "DIA_DUTY",
    })

    validos = validos.sort_values(["_pk", "DIA_DUTY"])[HEADERS_WB + ["_pk"]]

    piernas_por_pk = validos.groupby("_pk").size()
    pk_incompletos = piernas_por_pk[piernas_por_pk < 2].index
    incompletos = validos[validos["_pk"].isin(pk_incompletos)].copy()
    validos = validos[~validos["_pk"].isin(pk_incompletos)].copy()

    return validos, incompletos


validos_todas_flotas, incompletos = cargar_y_filtrar_wb(df_raw, MES_OBJETIVO, ANIO_OBJETIVO)
validos = validos_todas_flotas[validos_todas_flotas["AC Type"].astype(str).isin(cfg["subfleets"])].copy()
print(f"Pairings válidos de la flota elegida ({FLOTA_WB}): {validos['_pk'].nunique()}")


## 4. Leer la Matriz real y catálogo de instructores (idéntico a la Fase 3 de WB)

In [ ]:
URL_MATRIZ = "https://docs.google.com/spreadsheets/d/19WmwaoLDZnArNztu_dJNwi7bjrGq-_cx_gw0aN96zfk/edit?gid=580414308"
URL_ROL_INSTRUCTORES = "https://docs.google.com/spreadsheets/d/1yMvgb_O4qxpCCE4bAqD4XhW2GQI9ZObf2EAnymXaReE/edit?gid=1933640306"
URL_ARCHIVO_10_WB = "https://docs.google.com/spreadsheets/d/1NZN565fOJUtoETQvvPzdHpRvyHp4stY2hsrqtjNjSEU/edit"

sh_matriz = gc.open_by_url(URL_MATRIZ)
ws_matriz = sh_matriz.get_worksheet_by_id(580414308)

sh_rol_ins = gc.open_by_url(URL_ROL_INSTRUCTORES)
ws_rol_ins = sh_rol_ins.get_worksheet_by_id(1933640306)

sh_archivo10 = gc.open_by_url(URL_ARCHIVO_10_WB)
ws_archivo10_wb = sh_archivo10.get_worksheet_by_id(cfg["archivo10_gid"])
if ws_archivo10_wb.title != cfg["archivo10_tab_esperada"]:
    print(f"AVISO: esperaba la pestaña '{cfg['archivo10_tab_esperada']}' pero el gid abrió '{ws_archivo10_wb.title}'.")

FILA_ENCABEZADO_FECHAS = 2
FILA_PRIMER_INSTRUCTOR = 3
COL_PRIMERA_FECHA = 3

valores_m = ws_matriz.get_all_values()
fila_fechas = valores_m[FILA_ENCABEZADO_FECHAS - 1]
fechas_matriz = [c.strip() if c.strip() else None for c in fila_fechas[COL_PRIMERA_FECHA - 1:]]

reservas = []
fila_bp_a_matriz = {}
for i, fila in enumerate(valores_m[FILA_PRIMER_INSTRUCTOR - 1:]):
    if not fila or not fila[0].strip():
        continue
    bp = fila[0].strip().lstrip("'")
    nombre = fila[1].strip() if len(fila) > 1 else ""
    fila_bp_a_matriz[bp] = FILA_PRIMER_INSTRUCTOR + i
    for j, celda in enumerate(fila[COL_PRIMERA_FECHA - 1:]):
        if celda.strip().upper() == cfg["actividad_matriz"].upper():
            fecha_str = fechas_matriz[j] if j < len(fechas_matriz) else None
            if fecha_str:
                reservas.append((bp, nombre, fecha_str))

print(f"Slots '{cfg['actividad_matriz']}' encontrados en la Matriz: {len(reservas)}")

ROL_COL_IDE = {"767": "IDE B767", "787": "IDE B787"}[FLOTA_WB]
registros_rol = ws_rol_ins.get_all_records()
df_rol = pd.DataFrame(registros_rol)
instructores_ide_df = df_rol[df_rol[ROL_COL_IDE].astype(str).str.strip().str.upper() == "OK"].copy()
INSTRUCTORES_DATA_WB = list(zip(instructores_ide_df["Nombre"], instructores_ide_df.iloc[:, 0].astype(str)))
print(f"Instructores {ROL_COL_IDE} = OK: {len(INSTRUCTORES_DATA_WB)}")


## 5. Emparejar reservas con pairings reales (idéntico a la Fase 3 de WB)

In [ ]:
def normalizar_nombre(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.strip().lower()


def normalizar_fecha(fecha_str):
    d, m, a = fecha_str.strip().split("/")
    return (int(d), int(m), int(a))


def armar_pairings_dict(validos: pd.DataFrame):
    pairings = {}
    for pk, grupo in validos.groupby("_pk", sort=False):
        filas = [row for _, row in grupo.sort_values("DIA_DUTY").iterrows()]
        if len(filas) != 2:
            continue
        pairings[pk] = (filas[0], filas[1])
    return pairings


pairings_por_pk = armar_pairings_dict(validos)
pairings_por_fecha_ida = defaultdict(list)
for pk, (ida, _vta) in pairings_por_pk.items():
    pairings_por_fecha_ida[normalizar_fecha(ida["FECHA REAL"])].append(pk)


def emparejar_matriz_con_pairings(reservas, pairings_por_pk, pairings_por_fecha_ida):
    mapa_nombre_canonico = {normalizar_nombre(n): n for n, _bp in INSTRUCTORES_DATA_WB}
    usados_pk = set()
    asignaciones_por_pk = {}
    sin_bloque = []
    sin_match_nombre = []

    for bp, nombre_matriz, fecha_str in reservas:
        nombre_canonico = mapa_nombre_canonico.get(normalizar_nombre(nombre_matriz))
        if nombre_canonico is None:
            sin_match_nombre.append((bp, nombre_matriz, fecha_str))
            continue
        fecha_norm = normalizar_fecha(fecha_str)
        candidatos = [pk for pk in pairings_por_fecha_ida.get(fecha_norm, []) if pk not in usados_pk]
        if not candidatos:
            sin_bloque.append((bp, nombre_canonico, fecha_str))
            continue
        pk_elegido = candidatos[0]
        usados_pk.add(pk_elegido)
        ida, vta = pairings_por_pk[pk_elegido]
        fecha_vuelta_real = vta["FECHA REAL"]
        fecha_vuelta_distinta = normalizar_fecha(fecha_vuelta_real) != fecha_norm
        asignaciones_por_pk[pk_elegido] = (
            bp, nombre_canonico, cfg["actividad_matriz"],
            fecha_vuelta_real if fecha_vuelta_distinta else None,
        )
    return asignaciones_por_pk, sin_bloque, sin_match_nombre


asignaciones_por_pk, sin_bloque, sin_match_nombre = emparejar_matriz_con_pairings(
    reservas, pairings_por_pk, pairings_por_fecha_ida)

n_mia_multi_dia = sum(1 for *_r, fv in asignaciones_por_pk.values() if fv is not None)
print(f"Asignadas a un pairing real: {len(asignaciones_por_pk)} (de las cuales MIA multi-día: {n_mia_multi_dia})")
print(f"Sin pairing disponible esa fecha (revisar a mano): {len(sin_bloque)}")
print(f"Nombres sin match en el catálogo (revisar a mano): {len(sin_match_nombre)}")


## 6. Vista previa: texto real que reemplazaría el placeholder en la Matriz

Mismo formato de texto multilínea que NB (actividad / ruta / vuelo ida (STD-STA) / vuelo vuelta
(STD-STA)). Para un pairing LIM-MIA-LIM (ida y vuelta en fechas distintas), este MISMO texto
completo se escribe en las DOS celdas (día de ida y día de vuelta) — se repite el contexto
completo del pairing en ambos días porque no hay una regla dada sobre mostrar solo la mitad;
**esto es un supuesto, confirmar con Fernando la primera vez que se vea el resultado real.**

In [ ]:
def construir_texto_matriz(ida, vta, actividad):
    ruta = f'{ida["Dep Stn"]}-{ida["Arr Stn"]}-{vta["Arr Stn"]}'
    linea_ida = f'LA {ida["Flight No"]} ({str(ida["STD"])[:5]}-{str(ida["STA"])[:5]} hrs)'
    linea_vta = f'LA {vta["Flight No"]} ({str(vta["STD"])[:5]}-{str(vta["STA"])[:5]} hrs)'
    return f"{actividad}\n{ruta}\n{linea_ida}\n{linea_vta}"


preview_rows = []
celdas_a_escribir = []  # (bp, fecha_str, texto) -> para la celda de escritura real

for pk, (bp, nombre, actividad, fecha_vuelta) in asignaciones_por_pk.items():
    ida, vta = pairings_por_pk[pk]
    texto = construir_texto_matriz(ida, vta, actividad)
    fila_matriz = fila_bp_a_matriz.get(bp)
    if fila_matriz is None:
        print(f"AVISO: no encontré la fila de {nombre} (BP {bp}) en la Matriz, se salta.")
        continue

    preview_rows.append({"BP": bp, "Instructor": nombre, "Fecha celda": ida["FECHA REAL"],
                          "Texto": texto, "2da celda (vuelta)": fecha_vuelta or ""})
    celdas_a_escribir.append((fila_matriz, ida["FECHA REAL"], texto))
    if fecha_vuelta is not None:
        celdas_a_escribir.append((fila_matriz, fecha_vuelta, texto))
        preview_rows.append({"BP": bp, "Instructor": nombre, "Fecha celda": fecha_vuelta,
                              "Texto": texto, "2da celda (vuelta)": "(esta es la 2da celda)"})

df_preview_matriz = pd.DataFrame(preview_rows)
print(f"=== VISTA PREVIA: {len(celdas_a_escribir)} celdas a escribir en la Matriz ===")
display(df_preview_matriz)


## 7. Escribir en la Matriz — DESACTIVADO por defecto

Revisa primero la vista previa de la celda anterior.

In [ ]:
# --- DESCOMENTAR SOLO DESPUÉS DE REVISAR LA VISTA PREVIA ---

# celdas_gspread = []
# for fila_matriz, fecha_str, texto in celdas_a_escribir:
#     fecha_norm = normalizar_fecha(fecha_str)
#     col_idx = None
#     for j, f in enumerate(fechas_matriz):
#         if f is not None and normalizar_fecha(f) == fecha_norm:
#             col_idx = COL_PRIMERA_FECHA + j
#             break
#     if col_idx is None:
#         print(f"AVISO: la fecha {fecha_str} no está en las columnas de la Matriz, se salta esa celda.")
#         continue
#     celdas_gspread.append(gspread.Cell(row=fila_matriz, col=col_idx, value=texto))
#
# if celdas_gspread:
#     ws_matriz.update_cells(celdas_gspread, value_input_option="USER_ENTERED")
#     print(f"Escritas {len(celdas_gspread)} celdas en la Matriz (reemplazando el placeholder).")


## 8. CUADRO FINAL en Archivo 10 — búsqueda dinámica de columnas

Confirmado que el orden de columnas es DISTINTO por flota (767: fecha, día, ruta, vuelo, cupo,
INS, Grupo · 787: Fecha, DíaSEM, Vuelo, Ruta, N° Cupos, Grupos, INS) — se busca cada columna
por su nombre de encabezado (normalizando tildes/° /mayúsculas), no por posición fija, así este
mismo código sirve para ambas pestañas sin cambios.

Una fila por VUELO (no por pairing) — mismo criterio que NB (manual 2.18: "se pega la fecha, el
día, el vuelo, la ruta y los cupos de cada vuelo seleccionado"). **Grupo se deja en blanco**
(misma regla ya confirmada para NB: el grupo se resuelve aparte, con una fórmula/tabla viva).

In [ ]:
def normalizar_header(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.strip().lower()


COLUMNAS_CUADRO_FINAL = {
    "fecha": ["fecha"],
    "dia": ["dia", "sem"],
    "vuelo": ["vuelo"],
    "ruta": ["ruta"],
    "cupos": ["cupo"],
    "ins": ["ins"],
    "grupo": ["grupo"],
}


def buscar_fila_headers_y_mapeo(valores, columnas_esperadas):
    for i, fila in enumerate(valores):
        celdas_norm = [normalizar_header(c) for c in fila]
        mapeo = {}
        for clave, keywords in columnas_esperadas.items():
            for j, cn in enumerate(celdas_norm):
                if cn and any(kw in cn for kw in keywords):
                    mapeo[clave] = j
                    break
        if len(mapeo) == len(columnas_esperadas):
            return i, mapeo
    return None, None


valores_a10 = ws_archivo10_wb.get_all_values()
fila_header_cf_idx, mapeo_cf = buscar_fila_headers_y_mapeo(valores_a10, COLUMNAS_CUADRO_FINAL)

if fila_header_cf_idx is None:
    raise RuntimeError(
        f"No encontré una fila con las 7 columnas del CUADRO FINAL (fecha/día/vuelo/ruta/cupos/ins/grupo) "
        f"en '{ws_archivo10_wb.title}' -> revisar encabezados reales de la hoja."
    )

print(f"Fila de encabezado CUADRO FINAL detectada: fila {fila_header_cf_idx + 1} (1-indexado)")
print("Columnas detectadas (0-indexado):", mapeo_cf)


def cupos_wb():
    return cfg["cupos_por_vuelo"]


filas_cuadro_final = []
for pk, (bp, nombre, actividad, fecha_vuelta) in asignaciones_por_pk.items():
    ida, vta = pairings_por_pk[pk]
    for leg in (ida, vta):
        filas_cuadro_final.append({
            "fecha": leg["FECHA REAL"], "dia": leg["Day of Week"], "vuelo": leg["Flight No"],
            "ruta": f'{leg["Dep Stn"]}-{leg["Arr Stn"]}', "cupos": cupos_wb(),
            "ins": nombre, "grupo": "",
        })

df_cuadro_final = pd.DataFrame(filas_cuadro_final)
print(f"\nFilas para el CUADRO FINAL ({len(filas_cuadro_final)} vuelos, {len(asignaciones_por_pk)} pairings):")
display(df_cuadro_final)


## 9. Escribir el CUADRO FINAL — DESACTIVADO por defecto

Escribe cada columna en su posición real (detectada arriba), fila por fila, empezando justo
debajo del encabezado detectado.

In [ ]:
# --- DESCOMENTAR SOLO DESPUÉS DE REVISAR LA VISTA PREVIA ---

# fila_datos_ini = fila_header_cf_idx + 2  # 1-indexado, justo debajo del encabezado
# celdas_cf = []
# for i, fila_dict in enumerate(filas_cuadro_final):
#     fila_hoja = fila_datos_ini + i
#     for clave, col_idx0 in mapeo_cf.items():
#         valor = fila_dict[clave]
#         celdas_cf.append(gspread.Cell(row=fila_hoja, col=col_idx0 + 1, value=valor))
#
# if celdas_cf:
#     ws_archivo10_wb.update_cells(celdas_cf, value_input_option="USER_ENTERED")
#     print(f"Escritas {len(celdas_cf)} celdas del CUADRO FINAL en '{ws_archivo10_wb.title}'.")


## 10. QA: recalcular en Python que todo quedó consistente

In [ ]:
nombres_catalogo = {n for n, _bp in INSTRUCTORES_DATA_WB}
malos = [(pk, n) for pk, (_bp, n, _a, _fv) in asignaciones_por_pk.items() if n not in nombres_catalogo]
print("Asignaciones con nombre fuera del catálogo (deben ser 0):", len(malos))

print("Pairings asignados (únicos por construcción):", len(asignaciones_por_pk))

celdas_totales_esperadas = len(asignaciones_por_pk) + sum(1 for *_r, fv in asignaciones_por_pk.values() if fv is not None)
print(f"Celdas de Matriz a escribir: {len(celdas_a_escribir)} (esperado: {celdas_totales_esperadas})")
assert len(celdas_a_escribir) == celdas_totales_esperadas

print(f"Filas del CUADRO FINAL: {len(filas_cuadro_final)} (esperado: 2 x {len(asignaciones_por_pk)} pairings = {2 * len(asignaciones_por_pk)})")
assert len(filas_cuadro_final) == 2 * len(asignaciones_por_pk)

cupos_malos = [f for f in filas_cuadro_final if f["cupos"] != cfg["cupos_por_vuelo"]]
print(f"Filas con cupos distinto de {cfg['cupos_por_vuelo']} (deben ser 0):", len(cupos_malos))
assert not cupos_malos

grupo_no_vacio = [f for f in filas_cuadro_final if f["grupo"] != ""]
print("Filas con Grupo NO vacío (debe ser 0, se llena aparte):", len(grupo_no_vacio))
assert not grupo_no_vacio

fuera_de_flota = validos[~validos["AC Type"].astype(str).isin(cfg["subfleets"])]
print(f"Filas coladas de otra subflota (deben ser 0): {len(fuera_de_flota)}")
assert len(fuera_de_flota) == 0


## 11. Estado y pendientes — sin inventar nada

### Lo que se automatiza en este notebook
- Reconstruye los pairings reales (misma lógica ya validada en la Fase 3 de WB) y los cruza con
  las reservas de la Matriz.
- Reemplaza el placeholder `"LCK B767"`/`"LCK B787"` por el texto real del vuelo, y **cuando el
  pairing es LIM-MIA-LIM, escribe también la segunda celda** (fecha de vuelta) — cerrando el
  punto que había quedado pendiente desde la Fase 1/2 (confirmado por Fernando: ambos días se
  marcan).
- CUADRO FINAL en Archivo 10, con búsqueda dinámica de columnas por nombre de encabezado (no
  posición fija) porque el orden de columnas difiere entre "prueba de LCK 767" y
  "prueba de LCK 787" — una fila por vuelo, Grupo siempre en blanco (misma regla que NB).

### Confirmado por Fernando (2026-09-24)
- CUADRO FINAL ya existe en ambas pestañas "prueba de LCK 767"/"prueba de LCK 787", con headers
  en la fila 4 (767: AO-AU `fecha,día,ruta,vuelo,cupo,INS,Grupo`; 787: AP-AV
  `Fecha,DíaSEM,Vuelo,Ruta,N° Cupos,Grupos,INS`).

### Supuesto que hice y que hay que confirmar la primera vez que corra contra los Sheets reales
- **Texto de la segunda celda (día de vuelta) de un pairing MIA:** se repite el mismo texto
  completo (actividad + ruta + ambos vuelos) que en la celda de ida, porque no hay una regla
  dada sobre mostrar algo distinto ahí — revisar contra tu criterio la primera vez que se vea
  un caso real antes de confiar en el formato.

### Lo que sigue sin automatizar — falta información para no inventarlo
- **Freeze (roster + tabla alterna CUADRO FINAL, "las dos tablas"):** confirmaste que va en el
  mismo archivo "202609 Freeze LP", pestañas "LCK 767"/"LCK B787", pero todavía falta el
  link/`gid` exacto de esas pestañas para escribir ahí (igual patrón que se hizo para NB, solo
  falta el destino).
- **"INS F a considerar"/"Grupo" por tripulante (Y→Z/AA):** en tu captura de "prueba de LCK 787"
  aparecen "Grupo 1"/"Karla y Cris"/"Grupo 2"/"Sebas"/"Grupo 3"/"Fio" en un layout que parece
  distinto al de NB (`AD2..AG2`, una celda por grupo) — antes de automatizar esto para WB
  necesito que confirmes las celdas exactas donde están esos grupos (y su formato: ¿una celda
  por grupo con nombres separados por coma, como NB, o pares etiqueta/valor como se ve en la
  captura?) y dónde está la columna equivalente a "INS FINAL" (Y en NB) por tripulante en estas
  hojas WB.
- PDR / descanso reglamentario exacto contra vuelos reales de línea del instructor — no se
  calcula, igual que en NB y en la Fase 3 de WB.
